# 最新代码跨日期验证审计

## tl;dr

20260601、20260706、20260806 的六个全天样本任务均已完成：每市六标的，合计 172,835 帧匹配，0 不匹配、0 数据错误、0 缺少来源；另有 4 项停牌 PreOpen 排除。全量任务仍在运行，样本结果不代表三日全市场通过。

## Context & Methods

验证日期：20260601（用户指定）、20260706（当前实现的沪市 ETF 收盘规则分界日）、20260806（此前固定种子抽取）。每个日期 SH/SZ 股票和 ETF 全市场回放，与 raw snapshot 比较。另用同一冻结二进制每市六标的做全天样本，单独记录，不计入全量。

### Key Assumptions

- 全量 `--include-etfs`，没有 `--symbols`；样本的明确标的列表保存在每个命令回执。
- SZ 实用市价策略 `RestAtLastTradePrice`；U 空簿仍按现有严格撤单对账。没有为本轮结果修改 Rust 算法、字段或窗口。
- 可比帧匹配率 = matched / (matched + mismatched)；停牌排除、数据错误、缺少来源分开统计。中止没有全日匹配率。
- binary 和实际 Rust 源文件均冻结；仅 HEAD 无法描述有未提交修改的源码，审计同时验证源文件哈希。
- SZ 的 standard_acceptance=false 是实用策略标记，不是字段不匹配；快照匹配仍不证明逐事件中间态、全部 FIFO 路径或所有日期正确。
- 本 notebook 仅读已有报告，不重扫全市场原始数据，不写回 snapshot。

## Data

### 1. 加载与验证证据

Python 标准库即可审计；可从项目根目录或 analysis/ 执行。完整批次门槛通过 REQUIRE_COMPLETE=True 启用；未完成会明确失败。

In [1]:
from pathlib import Path
import json, runpy
ROOT = Path.cwd()
if not (ROOT / 'Cargo.toml').exists():
    ROOT = ROOT.parent
OUT = ROOT / 'reports/20260906-latest-cross-date-full'
REQUIRE_COMPLETE = False
audit = runpy.run_path(str(ROOT / 'analysis/audit_latest_cross_date_validation.py'))['audit']
result = audit(OUT, require_complete=REQUIRE_COMPLETE)
manifest = json.loads((OUT / 'manifest.json').read_text())
print({k: result[k] for k in ('as_of', 'binary_sha256', 'current_source_same', 'driver_status', 'heartbeat_age_seconds')})
for day in manifest['days']:
    print(day, {s['feed']: s['rows'] for s in manifest['sources'] if s['date'] == day})


{'as_of': '2026-09-06T14:18:45.693121+00:00', 'binary_sha256': 'b78c3270762e1b0f5e2deeb1eef40a54d03b43015b55c599291049b4c77f0ed9', 'current_source_same': True, 'driver_status': 'running', 'heartbeat_age_seconds': 3.801615}
20260601 {'mdl_4_24_0': 242093440, 'MarketData': 17426623, 'mdl_6_33_0': 178932534, 'mdl_6_36_0': 163061290, 'mdl_6_28_0': 16119011}
20260706 {'mdl_4_24_0': 246199689, 'MarketData': 14620240, 'mdl_6_33_0': 187223478, 'mdl_6_36_0': 170452634, 'mdl_6_28_0': 16100645}
20260806 {'mdl_4_24_0': 238205940, 'MarketData': 14566747, 'mdl_6_33_0': 177974699, 'mdl_6_36_0': 161989323, 'mdl_6_28_0': 16050997}


## Results

### 2. 全量与样本分开报告

每个 receipt 的输入行数已与相应源 Parquet footer 对账；所有分组计数均与总数对账，无诊断窗口覆盖。

In [2]:
print('All full jobs finished:', result['all_full_jobs_finished'])
print('All full validations passed:', result['all_full_validations_passed'])
for scope in ('full', 'smoke'):
    print('SCOPE:', scope)
    for row in result['rows']:
        if row['scope'] != scope:
            continue
        keys = ('date', 'market', 'status', 'exit_code', 'matched', 'mismatched', 'excluded_by_status', 'data_errors', 'missing_source', 'symbols')
        print({k: row[k] for k in keys if k in row})


All full jobs finished: False
All full validations passed: False
SCOPE: full
{'date': '20260601', 'market': 'SH', 'status': 'running'}
{'date': '20260601', 'market': 'SZ', 'status': 'running'}
{'date': '20260706', 'market': 'SH', 'status': 'not_started'}
{'date': '20260706', 'market': 'SZ', 'status': 'not_started'}
{'date': '20260806', 'market': 'SH', 'status': 'not_started'}
{'date': '20260806', 'market': 'SZ', 'status': 'not_started'}
SCOPE: smoke
{'date': '20260601', 'market': 'SH', 'status': 'completed', 'exit_code': 0, 'matched': 33055, 'mismatched': 0, 'excluded_by_status': 1, 'data_errors': 0, 'missing_source': 0, 'symbols': 6}
{'date': '20260601', 'market': 'SZ', 'status': 'completed', 'exit_code': 0, 'matched': 28430, 'mismatched': 0, 'excluded_by_status': 0, 'data_errors': 0, 'missing_source': 0, 'symbols': 6}
{'date': '20260706', 'market': 'SH', 'status': 'completed', 'exit_code': 0, 'matched': 29472, 'mismatched': 0, 'excluded_by_status': 0, 'data_errors': 0, 'missing_sourc

### 3. 分阶段、分品种质量与异常

每个已完成报告分别核对开盘后、盘中和收盘。聚合计数完整，异常明细最多 5000 条。

In [3]:
for row in result['rows']:
    if 'breakdown' not in row:
        if row.get('error'):
            print(row['date'], row['market'], row['scope'], 'ABORT:', row['error'][-1500:])
        continue
    print(row['date'], row['market'], row['scope'], 'match_rate', row['match_rate'])
    for group, counts in row['breakdown'].items():
        print(group, 'matched/comparable', counts['matched'], '/', counts['comparable'],
              'excluded/errors/missing', counts['excluded_by_status'], counts['data_errors'], counts['missing_source'])
    print('mismatch fields:', row['mismatch_fields'], 'tags:', row['match_tags'])
    print('exclusion/error reasons:', row['not_comparable_reasons'])
    for anomaly in row['sample_anomalies'][:3]:
        print('anomaly:', anomaly)


20260601 SH smoke match_rate 1.0
etf.continuous_trading matched/comparable 18283 / 18283 excluded/errors/missing 0 0 0
etf.market_close matched/comparable 3 / 3 excluded/errors/missing 0 0 0
etf.pre_open matched/comparable 2 / 2 excluded/errors/missing 1 0 0
stock.continuous_trading matched/comparable 14761 / 14761 excluded/errors/missing 0 0 0
stock.market_close matched/comparable 3 / 3 excluded/errors/missing 0 0 0
stock.pre_open matched/comparable 3 / 3 excluded/errors/missing 0 0 0
mismatch fields: {} tags: {}
exclusion/error reasons: {'excluded phase status SUSP': 1}
20260601 SZ smoke match_rate 1.0
etf.continuous_trading matched/comparable 9464 / 9464 excluded/errors/missing 0 0 0
etf.market_close matched/comparable 2 / 2 excluded/errors/missing 0 0 0
etf.pre_open matched/comparable 2 / 2 excluded/errors/missing 0 0 0
stock.continuous_trading matched/comparable 18954 / 18954 excluded/errors/missing 0 0 0
stock.market_close matched/comparable 4 / 4 excluded/errors/missing 0 0 0
st

## Takeaways

最新代码已通过上述三个日期的沪深样本；20260601 此前 SH 三个历史 ETF 收盘误拒和 SZ 市价余量中止未在本次样本重现。每市六标的、全天原始输入，不是全市场验收。

全量状态以 reports/20260906-latest-cross-date-full/summary.json 为准。六个 full 任务全部完成后重新从头执行并启用完整性门槛。即便全部快照通过，也不证明每个逐事件中间态或 FIFO 路径正确。旧 running 字段不能替代实时心跳与任务回执。